In [1]:
from utilities import init_bigquery_client
from google.cloud import bigquery
import os
import pandas as pd
import numpy as np
import plotly.express as px

from plotnine import *

#init BigQuery client
bq = init_bigquery_client()

Using BigQuery credentials: etl-testing-478716-c0b6c2c512e0.json


## Bulk Queries

In [2]:
# Read from the 'events' table in BigQuery
query = """
    SELECT *
    FROM `etl-testing-478716.firebase_etl_prod.events`
"""
events_df = bq.query(query).to_dataframe()

# Read from the 'userinvites' table in BigQuery
query = """
    SELECT *
    FROM `etl-testing-478716.firebase_etl_prod.userinvites`
"""
userinvites_df = bq.query(query).to_dataframe()

# Read from the 'users' table in BigQuery
query = """
    SELECT *
    FROM `etl-testing-478716.firebase_etl_prod.users`
"""
users_df = bq.query(query).to_dataframe()

/opt/miniconda3/envs/heyyall/lib/python3.13/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.


In [32]:
#Creating previous joiners df for network analysis
userinvites_df.rename(columns={'user_id': 'invite_user_id',
                               'createdAt': 'invite_created_at'}, inplace=True)

events_df.rename(columns={'user_id': 'event_user_id',
                          'createdAt': 'event_created_at'}, inplace=True)

# Merge the 'events' and 'userinvites' DataFrames on the 'event_id' column
merged_df = pd.merge(events_df, userinvites_df, left_on = 'document_id', 
                     right_on='event_id', how='inner', suffixes=('_events', '_userinvites'))
keep_columns = ['event_id', 'title', 'invite_user_id','event_user_id','type','status', 'invite_created_at', 'event_created_at']

filtered_df = merged_df[keep_columns]
filtered_df = filtered_df[filtered_df['status'] == 'accepted']
filtered_df = filtered_df.drop_duplicates(subset=['event_id', 'invite_user_id', 'event_user_id','type','status'])

#getting count of how many times each user attended events planned by each event planner
previous_joiners_df = filtered_df.groupby(['invite_user_id', 'event_user_id']).size().reset_index(name='pair_count')

#merging on if event planner is a businses user
biz_user_df = users_df[['user_id', 'businessUser']]

previous_joiners_df = previous_joiners_df.merge(biz_user_df, left_on='event_user_id', right_on='user_id', how='left')
previous_joiners_df['businessUser'] = previous_joiners_df['businessUser'].fillna(False)

In [35]:
#getting span of time user has been attending events planned by each event planner
first_attend_df = filtered_df.groupby(['invite_user_id', 'event_user_id'])['invite_created_at'].min().reset_index(name='first_attend_date')
last_attend_df = filtered_df.groupby(['invite_user_id', 'event_user_id'])['invite_created_at'].max().reset_index(name='last_attend_date')

previous_joiners_df = previous_joiners_df.merge(first_attend_df, on=['invite_user_id', 'event_user_id'], how='left')
previous_joiners_df = previous_joiners_df.merge(last_attend_df, on=['invite_user_id', 'event_user_id'], how='left')

previous_joiners_df['attend_span'] = (previous_joiners_df['last_attend_date'] - previous_joiners_df['first_attend_date']).dt.seconds()

KeyError: 'last_attend_date'

In [34]:
previous_joiners_df.head()

,invite_user_id,event_user_id,pair_count,user_id,businessUser,first_attend_date,last_attend_date,attend_span
0,0PNBuWC4P5by27fmCRYtT89Jdtl1,7QiegKPyi1fArNWHTstwm8WjPct1,1,7QiegKPyi1fArNWHTstwm8WjPct1,False,2026-01-07 19:41:43.782000+00:00,2026-01-07 19:41:43.782000+00:00,0
1,0PNBuWC4P5by27fmCRYtT89Jdtl1,9gHK6nfBJrVyx1543vFgvOKL85N2,1,9gHK6nfBJrVyx1543vFgvOKL85N2,False,2025-12-13 03:29:11.510000+00:00,2025-12-13 03:29:11.510000+00:00,0
2,0PNBuWC4P5by27fmCRYtT89Jdtl1,F1lO9Fp7s1TrCnbsiIwi0hvMZVq2,1,F1lO9Fp7s1TrCnbsiIwi0hvMZVq2,False,2025-11-19 21:22:13.230000+00:00,2025-11-19 21:22:13.230000+00:00,0
3,0PNBuWC4P5by27fmCRYtT89Jdtl1,QFxHzayNZSRyDcrLAPJf5Oi9dY42,4,QFxHzayNZSRyDcrLAPJf5Oi9dY42,False,2025-11-25 16:42:36.261000+00:00,2025-12-12 18:53:43.383000+00:00,17
4,0PNBuWC4P5by27fmCRYtT89Jdtl1,Z347q5fi8fUr1eeY2mt2W8rrmN92,1,Z347q5fi8fUr1eeY2mt2W8rrmN92,True,2025-12-04 15:43:01.549000+00:00,2025-12-04 15:43:01.549000+00:00,0
